# Lesson 2.1 · Scaling and transformations

A machine learning algorithm does not know what euros or years are: it only sees numbers. Scaling brings the features onto
comparable scales; transformations change the shape of their distribution. This lesson shows what
each tool does on loan application data, and when it is not needed at all.

**What you will learn**

- Which models need scaling and which don't, and why.
- What `fit` computes and what `transform` applies, in each scaler.
- How Standard, MinMax and Robust behave in the presence of outliers.
- How to reduce skewness with log, `PowerTransformer` and `QuantileTransformer`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats
from sklearn.preprocessing import (MaxAbsScaler, MinMaxScaler, PowerTransformer, QuantileTransformer,
                                   RobustScaler, StandardScaler)

df = ds.loans(with_missing=False)
df[["age", "income", "job_tenure", "amount"]].describe().round(1)

## Who needs scaling

| Model family | Why | Scaling |
|---|---|---|
| Distance-based (KNN, k-means, kernel SVM) | the Euclidean distance sums the squares of the differences: the feature with the largest unit dominates | **needed** |
| Gradient-based and penalized (regularized regression, neural networks) | the penalty and the speed of convergence depend on the unit | **needed** |
| PCA | it looks for the directions of maximum variance: the feature with the numerically largest variance wins | **needed** |
| Trees, forests, boosting | they compare one feature at a time with a threshold: the order of the values does not change | useless |
| Unregularized linear and logistic regression | rescaling a feature rescales the coefficient in the opposite direction | useless for predictions (useful for interpreting) |

## Three scalers, three ideas

```
StandardScaler   z = (x − mean) / standard deviation      mean 0, standard deviation 1
MinMaxScaler     z = (x − min) / (max − min)               everything in [0, 1]
RobustScaler     z = (x − median) / (q75 − q25)            ignores the extremes
```

Mean, standard deviation, minimum and maximum are **sensitive to outliers**; median and interquartile range are not.

In [ ]:
income = df[["income"]]
print("income: median", int(income.median().iloc[0]), "| maximum", int(income.max().iloc[0]))
tab = {}
for name, scaler in [("standard", StandardScaler()), ("minmax", MinMaxScaler()), ("robust", RobustScaler())]:
    z = scaler.fit_transform(income).ravel()
    tab[name] = {"typical value (median)": np.median(z), "outlier (maximum)": z.max(), "std deviation": z.std(),
                 "share of data in [-1, 1]": np.mean(np.abs(z) <= 1)}
pd.DataFrame(tab).round(2)

With `MinMaxScaler` the typical income ends up near 0.2 and all the others are squeezed into the same narrow interval
because a single huge value fixed the maximum. `RobustScaler` centers on the median and uses the interquartile range:
the typical values have a comparable spread, and the outlier stays an outlier (large value) without ruining the others.

## `fit`, `transform`, `fit_transform`

Every scaler **learns** some numbers with `fit` (mean and standard deviation, or minimum and maximum...) and **applies** them with
`transform`. The learned attributes have a trailing underscore.

In [ ]:
from sklearn.model_selection import train_test_split

tr, te = train_test_split(df[["age", "income"]], test_size=0.25, random_state=0)
sc = StandardScaler().fit(tr)
print("learned on the TRAINING set: mean", sc.mean_.round(1), "| scale", sc.scale_.round(1))
print("test with transform         :", sc.transform(te).mean(axis=0).round(3), " <- it is not zero, and that is right")
print("test with fit_transform     :", StandardScaler().fit_transform(te).mean(axis=0).round(3), " <- recomputed on the test: WRONG")

> **Common mistake.** `fit` only on the training set, `transform` on everything else. Refitting the scaler on the test set (or
> fitting it on the whole dataset before the split) is a leakage mistake that exercises E1 and E3 of module 1 and this
> module make you hunt for. Inside a `Pipeline` (module 3) it is impossible to do it.

## Changing the shape: log, Yeo-Johnson, quantiles

Scaling does not change the **shape** of a distribution. If a feature is strongly skewed (incomes, prices, counts)
and the model is linear or distance-based, it is worth transforming it:

- `np.log1p(x)`: simple, requires x ≥ 0 (or x > -1);
- `PowerTransformer(method="yeo-johnson")`: it chooses by itself the power that makes the distribution more symmetric;
- `QuantileTransformer(output_distribution="normal")`: it maps the quantiles onto a Gaussian. It is monotonic but it **squashes**
  the distances: two different extreme values end up almost equal.

In [ ]:
x = df[["income"]]
transformed = {
    "raw": x.to_numpy(),
    "log1p": np.log1p(x).to_numpy(),
    "yeo-johnson": PowerTransformer().fit_transform(x),
    "quantiles -> normal": QuantileTransformer(n_quantiles=200, output_distribution="normal").fit_transform(x),
}
pd.DataFrame({k: {"skewness": stats.skew(v.ravel()), "kurtosis": stats.kurtosis(v.ravel())}
              for k, v in transformed.items()}).round(2)

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>I have to train a Random Forest: do I standardize the features?</summary>

No. Every split compares a feature with a threshold and the order of the values does not change with a monotonic transformation: the chosen splits are identical, and scaling does not change the predictions. It costs time and complicates the pipeline.

</details>

<details><summary>I have a huge outlier in income. MinMaxScaler or RobustScaler?</summary>

RobustScaler: the maximum is exactly the value that ruins the MinMaxScaler, because it fixes the denominator. With the median and the interquartile range the typical values keep a sensible scale. If the outlier is a measurement error, it is better to correct it or remove it beforehand.

</details>